# relu-elementwise-max — worked example 1: Implement ReLU as Elementwise Maximum with Zero

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `relu-elementwise-max`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

ReLU (Rectified Linear Unit) is defined as `ReLU(x) = max(x, 0)` applied independently to every element of a tensor. Positive values pass through unchanged; negative values become zero. In PyTorch, `torch.maximum(x, torch.tensor(0.0))` implements this definition directly, making the operation's nature explicit compared to `F.relu`.

## Worked solution

**Step 1 — understand the operation.** For each element `x_i`, the output is `x_i` if `x_i > 0`, else `0`. This is a piecewise linear function with a kink at zero.

**Step 2 — use `t.maximum`.** `t.maximum(a, b)` returns the element-wise maximum between two tensors of the same shape. To implement ReLU, we broadcast a scalar zero: `t.maximum(x, t.tensor(0.0))`.

**Step 3 — apply to a test input.** Create a tensor with a mix of positive, negative, and zero values to make the behavior visible.

**Step 4 — compare positive/negative outputs.** Positive elements in `x` should appear unchanged in the output. Negative elements should become 0. The zero element stays 0.

**Step 5 — verify with `t.clamp` as ground truth.** `x.clamp(min=0)` is the equivalent operation; we check that both give the same result.

In [ ]:
import torch as t

def relu_via_maximum(x: t.Tensor) -> t.Tensor:
    """ReLU using the canonical elementwise-max definition."""
    return t.maximum(x, t.tensor(0.0))

# --- exercise and print ---
t.manual_seed(3)
x = t.tensor([-3.0, -1.5, 0.0, 1.2, 2.7, -0.5])
y = relu_via_maximum(x)

print('Input:  ', x.tolist())
print('ReLU:   ', y.tolist())
print('Expected zeros:', (x < 0).sum().item(), ' negative elements zeroed:', (y[x < 0] == 0).all().item())
print('Positive elements unchanged:', t.allclose(y[x > 0], x[x > 0]))

# Ground truth comparison
gt = x.clamp(min=0)
print('Matches clamp(min=0):', t.allclose(y, gt))